In [1]:
from scraper import fetch_website_links, fetch_website_contents
from dotenv import load_dotenv
load_dotenv(override=True)
from IPython.display import Markdown, display, update_display
import json
from openai import OpenAI




In [2]:

MODEL = "llama3.1:8b"
client = OpenAI(
    base_url="http://localhost:11434/v1",  # ✅ correct ollama port + /v1
    api_key="ollama"                        # ✅ can be any non-empty string
)

In [3]:
link_system_prompt = """ 
you are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relvant to include in the brouchre about the company,
such as links to another page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "Links": [
        {"type": "about page", "url":"https://full.url/goes/here/about"},
        {"type": "careers page", "url":"https://full.url/goes/here/careers"},
    ]
}
"""


In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} - 
Please decide which of these are relevant links for a brocher about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy Policy, email links.

Links (some might be relative links):
"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt 


In [5]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type":"json_object"}
    )
    result =  response.choices[0].message.content
    try:
        links = json.loads(result)
        print(f"Found {len(links['Links'])} relevant links")
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}\nRaw result: {result}")
        raise
    
    return links

#print(select_relevant_links("https://edwarddonner.com/"))


In [6]:
#select_relevant_links("https://huggingface.co")

In [7]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing page:\n\n{contents}\n## Relevant links:\n"
    for link in relevant_links['Links']:
        result += f"\n\n### Link: {link["type"]}\n"
        result += fetch_website_contents(link["url"])
    return result
#print(fetch_page_and_all_relevant_links("https://edwarddonner.com/"))

In [8]:
brochure_system_prompt = """ 
You are an assistant that analyzes the contents of several revelant pages from a company website
and creates a short brochure about the company for prospective customers,investors and recruits to read.
Respond in a markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [9]:
def get_brochure_user_prompt(websitename, url):
    user_prompt = f"""
You are looking at a company called: {websitename}
Here are the contents of its landing page and other relevant pages:
use this information to create a brochure about the company in markdown without code blocks. \n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Limit to 5k characters for now
    return user_prompt 


In [10]:
#get_brochure_user_prompt("edwarddonnar", "https://edwarddonner.com/" )

In [11]:
def create_brochure(websitename,url):
    response = client.chat.completions.create(
        model=MODEL,
        messages= [
            {"role":"system", "content":brochure_system_prompt},
            {"role":"user", "content":get_brochure_user_prompt(websitename,url)}
        ]
    )
    result = response.choices[0].message.content

    display(Markdown(result))

In [ ]:
create_brochure("Edward Donnar" , "https://edwarddonner.com/")

ConnectionError: HTTPSConnectionPool(host='edwarddonnar.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='edwarddonnar.com', port=443): Failed to resolve 'edwarddonnar.com' ([Errno 11001] getaddrinfo failed)"))